# Redrob preprocess — Colab GPU + Gemini

- **CUDA GPU** → fast BGE embeddings (~15–45 min for 100K)
- **Gemini API** → JD parse + archetype labels
- Outputs → **`artifacts/`** (download zip and copy to your local repo)

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Gemini (optional):** Set `GEMINI_JD_PARSE` / `GEMINI_LABELS` in the auth cell. ADC via Colab Secret `GOOGLE_ADC_JSON`.

In [ ]:
# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/ashokbugude/recruiter-candidate.git'
PROJECT_ROOT = Path('/content/recruiter-candidate')

if not (PROJECT_ROOT / 'rank.py').exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)

os.chdir(PROJECT_ROOT)
rev = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip()
assert (PROJECT_ROOT / 'rank.py').exists(), f'Project not found: {PROJECT_ROOT}'
print('Project root:', PROJECT_ROOT.resolve())
print('Git revision:', rev)

In [ ]:
!pip install -q -r colab/requirements.txt

In [ ]:
from pathlib import Path
candidates_path = PROJECT_ROOT / 'challenge' / 'candidates.jsonl'
if not candidates_path.exists():
    from google.colab import files
    print('Upload candidates.jsonl (~465 MB)')
    uploaded = files.upload()
    (PROJECT_ROOT / 'challenge').mkdir(parents=True, exist_ok=True)
    for name, data in uploaded.items():
        candidates_path.write_bytes(data)
    print('Saved', candidates_path)
else:
    print('Found', candidates_path, f'({candidates_path.stat().st_size // 1_048_576} MB)')

In [ ]:
GEMINI_JD_PARSE = True   # ~30s — Gemini Pro JD (recommended)
GEMINI_LABELS = False    # ~40-60h — 100K Flash labels (off by default)
FORCE = False            # True = rebuild existing artifacts

import json
import os
from pathlib import Path

ADC_PATH = PROJECT_ROOT / 'credentials' / 'adc.json'

if GEMINI_JD_PARSE or GEMINI_LABELS:
    ADC_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not ADC_PATH.exists():
        try:
            from google.colab import userdata
            adc = json.loads(userdata.get('GOOGLE_ADC_JSON'))
            ADC_PATH.write_text(json.dumps(adc), encoding='utf-8')
            print('Wrote ADC credentials from Colab Secret')
        except Exception:
            from google.colab import files
            print('Upload your ADC credentials JSON (authorized_user with refresh_token)')
            uploaded = files.upload()
            for name, data in uploaded.items():
                ADC_PATH.write_bytes(data)
            print('Saved', ADC_PATH)
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(ADC_PATH)
    os.environ.setdefault('GOOGLE_CLOUD_LOCATION', 'global')
    adc_data = json.loads(ADC_PATH.read_text(encoding='utf-8'))
    if adc_data.get('quota_project_id'):
        os.environ.setdefault('GOOGLE_CLOUD_PROJECT', adc_data['quota_project_id'])
    os.environ['REDROB_SKIP_GEMINI'] = '0'
    os.environ['REDROB_GEMINI_JD_PARSE'] = '1' if GEMINI_JD_PARSE else '0'
    os.environ['REDROB_GEMINI_LABELS'] = '1' if GEMINI_LABELS else '0'
    os.environ.setdefault('REDROB_GEMINI_FLASH_MODEL', 'gemini-3.5-flash')
    os.environ.setdefault('REDROB_GEMINI_PRO_MODEL', 'gemini-2.5-pro')
    print('Vertex AI:', os.environ.get('GOOGLE_CLOUD_PROJECT'), '@', os.environ.get('GOOGLE_CLOUD_LOCATION'))
    print('Gemini flags: jd_parse=%s labels=%s' % (GEMINI_JD_PARSE, GEMINI_LABELS))
else:
    os.environ['REDROB_SKIP_GEMINI'] = '1'
    os.environ['REDROB_GEMINI_JD_PARSE'] = '0'
    os.environ['REDROB_GEMINI_LABELS'] = '0'

In [ ]:
import sys
cmd = [sys.executable, 'colab/run_preprocess_gpu.py']
if FORCE:
    cmd.append('--force')
if GEMINI_JD_PARSE:
    cmd.append('--gemini-jd')
else:
    cmd.append('--no-gemini-jd')
if GEMINI_LABELS:
    cmd.append('--gemini-labels')
    if FORCE:
        cmd.append('--force-llm')
else:
    cmd.append('--no-gemini-labels')
!{' '.join(cmd)}

In [ ]:
# Quick artifact check (run after preprocess)
import json
from pathlib import Path

artifacts = PROJECT_ROOT / 'artifacts'
checks = {
    'jd_requirements.json': artifacts / 'jd_requirements.json',
    'gemini_tiers.parquet': artifacts / 'gemini_tiers.parquet',
    'candidate_features.parquet': artifacts / 'candidate_features.parquet',
    'bge_embeddings.npy': artifacts / 'bge_embeddings.npy',
    'faiss.index': artifacts / 'faiss.index',
    'bm25.pkl': artifacts / 'bm25.pkl',
}
for name, path in checks.items():
  status = 'OK' if path.exists() else 'MISSING'
  print(f'{status:7} {name}')

jd_path = artifacts / 'jd_requirements.json'
if jd_path.exists():
    jd = json.loads(jd_path.read_text(encoding='utf-8'))
    print('JD source:', jd.get('source', '?'))
if (artifacts / 'gemini_tiers.parquet').exists():
    import polars as pl
    g = pl.read_parquet(artifacts / 'gemini_tiers.parquet')
    src = g['label_source'].value_counts().sort('count', descending=True)
    print('Label sources:', dict(zip(src['label_source'].to_list(), src['count'].to_list())))

In [ ]:
from google.colab import files
zip_path = PROJECT_ROOT / 'colab' / 'artifacts_download.zip'
assert zip_path.exists(), 'Run preprocess first — zip is created by run_preprocess_gpu.py'
files.download(str(zip_path))